# Autograd & Backpropagation

Understand how PyTorch builds computation graphs and computes gradients automatically. Master the chain rule and why gradient accumulation is not a bug—it's a feature.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/autograd-backprop)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: Automatic Differentiation, the Graph, and requires_grad


In [ ]:
import torch

# 1. Enable gradient tracking
x = torch.tensor(2.0, requires_grad=True)
print(f"x.requires_grad: {x.requires_grad}")

# 2. Build computation graph
y = x * 3
z = y + 1
loss = z ** 2

print(f"\nComputation graph:")
print(f"x = 2.0")
print(f"y = x * 3 = {y.item()}")
print(f"z = y + 1 = {z.item()}")
print(f"loss = z^2 = {loss.item()}")

# 3. Backward pass
loss.backward()
print(f"\nAfter loss.backward():")
print(f"x.grad = {x.grad}")  # Should be 42 (chain rule: 2*z*3)

# 4. Demonstrate graph is freed after backward
print(f"\nGraph state after backward:")
print(f"loss.grad_fn is None: {loss.grad_fn is None}")  # Graph was freed

# 5. To backward again, rebuild the graph
x.grad.zero_()  # Clear old gradient
y = x * 3
z = y + 1
loss = z ** 2
loss.backward()
print(f"After rebuilding graph:")
print(f"x.grad = {x.grad}")

# 6. Demonstrate requires_grad effect on intermediate tensors
print(f"\nIntermediate tensor gradients:")
x = torch.tensor(2.0, requires_grad=True)
y = x + 1  # Intermediate tensor (not a leaf)
z = y ** 2
loss = z.mean()
loss.backward()
print(f"x.grad (leaf tensor): {x.grad}")  # Exists
print(f"y.grad (intermediate): {y.grad}")  # None (freed to save memory)

# 7. Retain intermediate gradients if needed
x.grad.zero_()
y = x + 1
y.retain_grad()  # Keep y's gradient
z = y ** 2
loss = z.mean()
loss.backward()
print(f"y.grad with retain_grad(): {y.grad}")  # Now it exists

# 8. Forward mode perspective: multiple paths to same variable
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x ** 2
z = y * 3  # All elements of z depend on all elements of x
loss = z.sum()
loss.backward()
print(f"\nMulti-element backward:")
print(f"x.grad = {x.grad}")  # [6.0, 12.0] (3*2*x for each element)

### Lesson example: Manual Forward & Backward—The Chain Rule, by Hand


In [ ]:
import torch
import math

print("=" * 60)
print("CHAIN RULE: Single Variable")
print("=" * 60)

# Example 1: f(x) = sin(x^2)
x_val = math.pi / 4
f_val = math.sin(x_val ** 2)
df_dx_manual = math.cos(x_val ** 2) * 2 * x_val

x = torch.tensor(x_val, requires_grad=True)
f = torch.sin(x ** 2)
f.backward()

print(f"\nf(x) = sin(x²) at x = π/4")
print(f"Manual: f = {f_val:.6f}, df/dx = {df_dx_manual:.6f}")
print(f"PyTorch: f = {f.item():.6f}, df/dx = {x.grad.item():.6f}")

print("\n" + "=" * 60)
print("CHAIN RULE: Multiple Variables (Partial Derivatives)")
print("=" * 60)

# Example 2: f(x, y) = x² * y + sin(y)
x_val, y_val = 2.0, math.pi / 4

# Manual calculation
df_dx_manual = 2 * x_val * y_val
df_dy_manual = x_val ** 2 + math.cos(y_val)

x = torch.tensor(x_val, requires_grad=True)
y = torch.tensor(y_val, requires_grad=True)
f = x ** 2 * y + torch.sin(y)
f.backward()

print(f"\nf(x, y) = x² * y + sin(y) at (x={x_val}, y=π/4)")
print(f"Manual: ∂f/∂x = {df_dx_manual:.6f}, ∂f/∂y = {df_dy_manual:.6f}")
print(f"PyTorch: ∂f/∂x = {x.grad.item():.6f}, ∂f/∂y = {y.grad.item():.6f}")

print("\n" + "=" * 60)
print("VANISHING GRADIENTS: Deep Chain")
print("=" * 60)

# Deep RNN-like structure with sigmoid
x = torch.tensor([1.0], requires_grad=True)
print(f"\nInitial x.grad will be: 1.0 (implicit)")
for i in range(10):
    x = torch.sigmoid(x * 0.5)

loss = x
loss.backward()
print(f"After 10 sigmoid(0.5x) layers:")
print(f"Gradient: {x.grad:.2e}")  # Vanishingly small
print(f"Why: sigmoid'(x) ≤ 0.25, so 0.25^10 ≈ 9e-7")

print("\n" + "=" * 60)
print("GRADIENT ACCUMULATION")
print("=" * 60)

x = torch.tensor([1.0, 2.0], requires_grad=True)

# First backward
y = x ** 2
y.sum().backward()
print(f"\nAfter 1st backward: x.grad = {x.grad}")

# Second backward (gradient accumulates!)
y = x ** 2
y.sum().backward()
print(f"After 2nd backward (no zero_grad): x.grad = {x.grad}")

# Clear and try again
x.grad.zero_()
y = x ** 2
y.sum().backward()
print(f"After zero_grad + backward: x.grad = {x.grad}")

print("\n" + "=" * 60)
print("SCALAR REQUIREMENT: .backward() needs scalar loss")
print("=" * 60)

x = torch.randn(5, requires_grad=True)
y = x ** 2

# ❌ This fails: y is not scalar
try:
    y.backward()
except RuntimeError as e:
    print(f"Error: {str(e)[:50]}...")

# ✅ This works: reduce to scalar
loss = y.sum()
loss.backward()
print(f"✓ After y.sum().backward(): x.grad = {x.grad[:3]}...")

---

## Exercise: Verify Autograd by Hand


Implement a function f(x, y) = x² * y + sin(y), compute gradients manually using calculus, then verify PyTorch autograd computes the same values.

**Requirements:**
- Define f(x, y) = x² * y + sin(y) in both manual math and PyTorch
- Compute ∂f/∂x and ∂f/∂y analytically on paper
- At point (x=2, y=π/4), evaluate the analytical gradients by hand
- Use PyTorch autograd to compute gradients at the same point
- Verify the values match (within floating-point tolerance 1e-5)
- Print a comparison table showing manual vs PyTorch results

**Hints on Partial Derivatives:**
- ∂f/∂x = ∂(x² * y) / ∂x = 2xy  (y is constant w.r.t. x)
- ∂f/∂y = ∂(x² * y) / ∂y + ∂(sin y) / ∂y = x² + cos(y)
- At (x=2, y=π/4):
  - ∂f/∂x = 2 * 2 * (π/4) = π ≈ 3.14159
  - ∂f/∂y = 4 + cos(π/4) ≈ 4 + 0.707 ≈ 4.707


In [ ]:
import torch
import math

# 1. Analytical solution (compute by hand, fill in)
x_val = 2.0
y_val = math.pi / 4

# ∂f/∂x = 2 * x * y
df_dx_analytical = None  # TODO: Compute using formula

# ∂f/∂y = x^2 + cos(y)
df_dy_analytical = None  # TODO: Compute using formula

print(f"Analytical gradients at (x={x_val}, y=π/4):")
print(f"  ∂f/∂x = {df_dx_analytical:.6f}")
print(f"  ∂f/∂y = {df_dy_analytical:.6f}")

# 2. Autograd solution
x = torch.tensor(x_val, requires_grad=True)
y = torch.tensor(y_val, requires_grad=True)

# f(x, y) = x^2 * y + sin(y)
f = x**2 * y + torch.sin(y)

f.backward()

print(f"\nAutograd gradients:")
print(f"  ∂f/∂x = {x.grad.item():.6f}")
print(f"  ∂f/∂y = {y.grad.item():.6f}")

# 3. Verify they match
print(f"\nVerification:")
print(f"  Match ∂f/∂x? {torch.allclose(torch.tensor(df_dx_analytical), x.grad, atol=1e-5)}")
print(f"  Match ∂f/∂y? {torch.allclose(torch.tensor(df_dy_analytical), y.grad, atol=1e-5)}")

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/autograd-backprop) for the solution and discussion.
